<a href="https://colab.research.google.com/github/Sophieli0113/ds2002-fa26/blob/main/notebooks/01-foundations/2026%2009%2018%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
df = df.assign(revenue=df['qty'] * df['price'])

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Total revenue: ${total_revenue:,.2f}")
print(f"Total units: {total_units}")
print(f"With {len(df)} orders, average revenue per order is ${total_revenue/len(df):,.2f}. "
      f"Total revenue reflects {total_units} units sold across all vendors and categories.")

Total revenue: $8,520.00
Total units: 783
With 400 orders, average revenue per order is $21.30. Total revenue reflects 783 units sold across all vendors and categories.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO
by_category = (df.groupby('category')
               .agg(revenue=('revenue', 'sum'))
               .round(2)
               .sort_values('revenue', ascending=False))
by_category['share_pct'] = (100 * by_category['revenue'] / df['revenue'].sum()).round(2)
by_category

,revenue,share_pct
category,,
Food,4293.0,50.39
Merch,1771.5,20.79
Drink,1554.0,18.24
RainGear,901.5,10.58


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO
vendor_stats = (df.groupby('vendor_id')
                 .agg(avg_revenue=('revenue', 'mean'),
                      order_count=('revenue', 'count'))
                 .round(2)
                 .sort_values('avg_revenue', ascending=False))

top_vendor = vendor_stats.index[0]
print(f"Highest average order revenue: {top_vendor} "
      f"(${vendor_stats.loc[top_vendor, 'avg_revenue']:.2f} "
      f"across {int(vendor_stats.loc[top_vendor, 'order_count'])} orders)")
vendor_stats

Highest average order revenue: V-01 ($22.60 across 94 orders)


,avg_revenue,order_count
vendor_id,,
V-01,22.60,94
V-18,21.75,108
V-05,20.58,93
V-10,20.31,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO
merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()
share = 100 * merch_revenue / df['revenue'].sum()
print(f"Merch share of revenue: {share:.1f}%")

Merch share of revenue: 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
baseline_rows = len(df)
baseline_revenue = df['revenue'].sum()

joined = df.merge(vendor_names, on='vendor_id', how='left',
                   validate='many_to_one', indicator=True)

print('rows:', len(joined), '(expected', baseline_rows, ')')
print('revenue:', joined['revenue'].sum(), '(expected', baseline_revenue, ')')
assert len(joined) == baseline_rows
assert joined['revenue'].sum() == baseline_revenue

missing = joined[joined['_merge'] == 'left_only']
print(f"\nunmatched vendor_id(s): {missing['vendor_id'].unique().tolist()} | "
      f"revenue at stake: ${missing['revenue'].sum():.2f}")

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor')
joined = joined.drop(columns='_merge')

rows: 400 (expected 400 )
revenue: 8520.0 (expected 8520.0 )

unmatched vendor_id(s): ['V-18'] | revenue at stake: $2349.00


**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO
pivot = pd.pivot_table(joined, index='vendor_name', columns='category',
                        values='revenue', aggfunc='sum', fill_value=0,
                        margins=True, margins_name='Total').round(2)
pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown vendor,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

(a) Merch and Food make up over 70% of total revenue. RainGear made only 10.58%, even on a rainy game day. The category people probably wanted most sold the least. For next game, I think these vendors should stock more food and merch since they're already driving sales. For RainGear, try bundling it near entry gates or rethink pricing, since 13 units barely moved the needle.

(b) Q5 and Q6 are the least trustworthy parts of this report because of a missing name. Vendor V-18 never matched to a name in the vendor list, and it's not a small case — it's $2,349, about 27.6% of all revenue. Rather than delete those orders and undercount sales, they were kept and labeled Unknown vendor. That means one row in the pivot table isn't a real, identifiable vendor. So any best vendor claim only holds for the three vendors we do know, the fourth is just a gap in the data.